# FLOW — Large-Scale Vocabulary Growth on Kaggle

**Geometric Causal Architecture** — weight-free, token-free reasoning.  
Knowledge stored as shape in a 104D Riemannian manifold.

This notebook feeds **multiple real-world corpora** through the Phase 7 VocabularyBuilder pipeline  
to grow the expression vocabulary from ~100 entries to **50,000–100,000** entries.

### Corpus mix — grounded in agentic capabilities

| Dataset | Domain | Why it matters |
|---|---|---|
| Simple English Wikipedia | General knowledge | Broad concept coverage, clean prose |
| OpenAssistant (oasst1) | Agent assistance / dialogue | Conversational patterns, Q&A structure |
| Alpaca (cleaned) | Instruction following | Imperative/procedural language |
| Glaive Function Calling v2 | Tool calling / function use | Structured action language, parameters |
| SlimOrca | Reasoning chains | Step-by-step explanation patterns |

| Metric | Before | Target |
|---|---|---|
| Vocabulary entries | ~100 | 50K–100K |
| Manifold concepts | ~137 | 10K–50K |
| Output fluency | Proof-of-concept | Coherent |

**Hardware:** Kaggle CPU (4 cores, 30 GB RAM)  
**Estimated runtime:** 2–4 hours  
**Output artifacts:** `vocabulary_large.npz`, `manifold_large.npz`

## 1. Setup — Clone repo & install dependencies

In [ ]:
# Clone the FLOW repository
!git clone https://github.com/United-Visions/FLOW.git

# Install dependencies (pure math — no ML frameworks)
!pip install numpy scipy networkx datasets -q

# Verify
import os
assert os.path.isdir("FLOW/src"), "FLOW repo not found"
print("✓ FLOW repository cloned successfully")

In [ ]:
# Add FLOW to Python path
import sys
sys.path.insert(0, "FLOW")

# Verify imports
from src.phase5 import GEOPipeline, PipelineResult
from src.vocabulary import VocabularyBuilder, VocabularyStore
from src.persistence import ManifoldSnapshot
from src.phase3.annealing_engine.experience import Experience
from src.phase1.expression.matcher import ExpressionEntry
import numpy as np
import time

print("✓ All FLOW imports successful")
print(f"  numpy {np.__version__}")

## 2. Build the GEOPipeline

In [ ]:
# Build the full C1-C7 pipeline
# T0=1.0 : high initial temperature for flexible placement
# lambda_=0.005 : slower cooling for large-scale ingestion
# T_floor=0.03  : lower floor to allow crystallisation over time

pipeline = GEOPipeline(
    T0=1.0,
    lambda_=0.005,
    T_floor=0.03,
    flow_seed=42,
)

print(f"✓ Pipeline built")
print(f"  Manifold dimension : {pipeline.manifold.DIM}")
print(f"  Seed concepts      : {pipeline.n_concepts}")
print(f"  Temperature        : {pipeline.temperature:.4f}")

## 3. Load corpora — multi-domain mix

We feed five complementary datasets that ground FLOW in:
- **General knowledge** — Wikipedia (broad concept coverage)
- **Agent assistance** — OpenAssistant conversations (Q&A, dialogue)
- **Instruction following** — Stanford Alpaca (imperative/procedural text)
- **Tool calling** — Glaive Function Calling (structured actions, parameters)
- **Reasoning chains** — SlimOrca (step-by-step explanations)

Each dataset contributes different linguistic patterns to the manifold geometry.

In [ ]:
from datasets import load_dataset

all_texts = []   # collect (text, source_tag) pairs for unified feeding

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3A. Wikipedia — Simple English (general knowledge)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Simple English Wikipedia...")
ds_wiki = load_dataset("wikipedia", "20220301.simple", split="train", trust_remote_code=True)
N_WIKI = 50_000
for i, article in enumerate(ds_wiki):
    if i >= N_WIKI:
        break
    text = article.get("text", "")
    if text and len(text) > 50:
        all_texts.append((text, "wikipedia"))
print(f"  ✓ Wikipedia: {len([t for t in all_texts if t[1]=='wikipedia']):,} articles")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3B. OpenAssistant (oasst1) — agent assistance / dialogue
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading OpenAssistant (oasst1)...")
ds_oasst = load_dataset("OpenAssistant/oasst1", split="train", trust_remote_code=True)
N_OASST = 30_000
for i, row in enumerate(ds_oasst):
    if i >= N_OASST:
        break
    text = row.get("text", "")
    if text and len(text) > 30:
        all_texts.append((text, "openassistant"))
print(f"  ✓ OpenAssistant: {len([t for t in all_texts if t[1]=='openassistant']):,} messages")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3C. Alpaca (cleaned) — instruction following
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Alpaca (cleaned)...")
ds_alpaca = load_dataset("yahma/alpaca-cleaned", split="train", trust_remote_code=True)
N_ALPACA = 50_000
for i, row in enumerate(ds_alpaca):
    if i >= N_ALPACA:
        break
    # Concatenate instruction + input + output for full context
    parts = []
    if row.get("instruction"):
        parts.append(row["instruction"])
    if row.get("input"):
        parts.append(row["input"])
    if row.get("output"):
        parts.append(row["output"])
    text = " ".join(parts)
    if text and len(text) > 30:
        all_texts.append((text, "alpaca"))
print(f"  ✓ Alpaca: {len([t for t in all_texts if t[1]=='alpaca']):,} instructions")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3D. Glaive Function Calling v2 — tool calling / function use
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading Glaive Function Calling v2...")
ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train", trust_remote_code=True)
N_GLAIVE = 50_000
for i, row in enumerate(ds_glaive):
    if i >= N_GLAIVE:
        break
    # Each row has system prompt + chat — extract all text
    text_parts = []
    if row.get("system"):
        text_parts.append(row["system"])
    if row.get("chat"):
        text_parts.append(row["chat"])
    text = " ".join(text_parts)
    if text and len(text) > 30:
        all_texts.append((text, "glaive_functions"))
print(f"  ✓ Glaive Functions: {len([t for t in all_texts if t[1]=='glaive_functions']):,} examples")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3E. SlimOrca — reasoning chains
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("Loading SlimOrca...")
ds_orca = load_dataset("Open-Orca/SlimOrca", split="train", trust_remote_code=True)
N_ORCA = 50_000
for i, row in enumerate(ds_orca):
    if i >= N_ORCA:
        break
    # SlimOrca uses 'conversations' list with role/value dicts
    convos = row.get("conversations", [])
    text = " ".join(turn.get("value", "") for turn in convos if isinstance(turn, dict))
    if text and len(text) > 30:
        all_texts.append((text, "slimorca"))
print(f"  ✓ SlimOrca: {len([t for t in all_texts if t[1]=='slimorca']):,} reasoning chains")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Summary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
from collections import Counter
source_counts = Counter(t[1] for t in all_texts)
print(f"\n{'='*60}")
print(f"Total texts loaded: {len(all_texts):,}")
for src, cnt in source_counts.most_common():
    print(f"  {src:20s} : {cnt:>8,}")
print(f"{'='*60}")

## 4. Configure VocabularyBuilder & feed all corpora

In [ ]:
# ── Configuration ──────────────────────────────────────────────
# Vocabulary ceiling — controls how many unique words we keep.
# Higher = more expressive output, but more RAM and longer build.
V_MAX = 50_000          # target vocabulary size
MIN_COUNT = 10           # minimum frequency for a word to be kept
WINDOW_SIZE = 5          # co-occurrence sliding window
N_CONTRAST_PASSES = 3    # C4 refinement passes over PMI matrix
TAU_SAME = 1.0           # PMI threshold for SAME judgment
TAU_DIFF = -0.5          # PMI threshold for DIFFERENT judgment
BATCH_SIZE = 512         # contrast judgment batch size

print(f"Configuration:")
print(f"  V_MAX            = {V_MAX:,}")
print(f"  MIN_COUNT        = {MIN_COUNT}")
print(f"  WINDOW_SIZE      = {WINDOW_SIZE}")
print(f"  N_CONTRAST_PASSES= {N_CONTRAST_PASSES}")
print(f"  Total texts      = {len(all_texts):,}")

In [ ]:
# Build the VocabularyBuilder
builder = VocabularyBuilder(
    manifold=pipeline.manifold,
    annealing_engine=pipeline._annealing,
    contrast_engine=pipeline._contrast_engine,
    window_size=WINDOW_SIZE,
    min_count=MIN_COUNT,
    v_max=V_MAX,
    tau_same=TAU_SAME,
    tau_diff=TAU_DIFF,
    batch_size=BATCH_SIZE,
    n_contrast_passes=N_CONTRAST_PASSES,
)

print("✓ VocabularyBuilder created")

In [ ]:
# ── Feed all corpora ──────────────────────────────────────────────
# Shuffled to interleave domains — prevents the manifold from
# crystallising on one domain before seeing others.

import random
random.seed(42)
random.shuffle(all_texts)

t0 = time.time()
REPORT_EVERY = 25_000
n_total = len(all_texts)

for i, (text, source) in enumerate(all_texts):
    builder.feed(text)
    
    if (i + 1) % REPORT_EVERY == 0:
        elapsed = time.time() - t0
        rate = (i + 1) / elapsed
        print(f"  [{i+1:>8,} / {n_total:,}]  "
              f"tokens: {builder.n_tokens_fed:>12,}  "
              f"({rate:.0f} texts/sec, {elapsed:.1f}s elapsed)")

elapsed = time.time() - t0
print(f"\n✓ All corpora ingested")
print(f"  Texts processed    : {n_total:,}")
print(f"  Total tokens       : {builder.n_tokens_fed:,}")
print(f"  Time               : {elapsed:.1f}s ({elapsed/60:.1f} min)")

## 5. Build vocabulary — PMI matrix → Word placement → Contrast → Templates

This runs the full Phase 7 pipeline:
1. **PMI matrix** from accumulated co-occurrence counts
2. **Word placement** on M(t) via C3 Annealing Engine
3. **Contrast refinement** via C4 (SAME/DIFFERENT judgments from PMI)
4. **Template building** — manifold positions → ExpressionEntry objects
5. **Save** to `.npz`

In [ ]:
# ── Build and save vocabulary ────────────────────────────────────
# This is the computationally intensive step.
# On Kaggle CPU: ~30min–2hr depending on vocabulary size.

OUTPUT_DIR = "/kaggle/working"    # Kaggle output directory
VOCAB_PATH = f"{OUTPUT_DIR}/vocabulary_large.npz"

print("Building vocabulary (this may take a while)...")
print("  Step 1: Building PMI matrix from co-occurrence counts...")
t0 = time.time()

n_entries = builder.build_and_save(VOCAB_PATH)

elapsed = time.time() - t0
print(f"\n✓ Vocabulary built and saved")
print(f"  Entries written    : {n_entries:,}")
print(f"  Words placed on M(t): {builder.n_words_placed:,}")
print(f"  Contrast judgments : {builder.n_judgments_applied:,}")
print(f"  Build time         : {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"  File               : {VOCAB_PATH}")
print(f"  File size          : {os.path.getsize(VOCAB_PATH) / 1024 / 1024:.2f} MB")

In [ ]:
# ── Save manifold state ──────────────────────────────────────────
MANIFOLD_PATH = f"{OUTPUT_DIR}/manifold_large.npz"

n_saved = ManifoldSnapshot.save(pipeline.manifold, MANIFOLD_PATH)

print(f"✓ Manifold state saved")
print(f"  Concepts on M(t)   : {n_saved:,}")
print(f"  File               : {MANIFOLD_PATH}")
print(f"  File size          : {os.path.getsize(MANIFOLD_PATH) / 1024 / 1024:.2f} MB")
print(f"  Temperature        : {pipeline.temperature:.6f}")

## 6. Verify artifacts — load and validate

In [ ]:
# ── Verify manifold round-trip ───────────────────────────────────
info = ManifoldSnapshot.info(MANIFOLD_PATH)
print("Manifold snapshot info:")
for k, v in info.items():
    print(f"  {k:20s} : {v}")

# Spot-check: reload and compare positions
reloaded = ManifoldSnapshot.load(MANIFOLD_PATH)
labels = list(pipeline.manifold._points.keys())
max_err = 0.0
for label in labels[:100]:   # check first 100
    orig = pipeline.manifold.position(label)
    rest = reloaded.position(label)
    err = np.max(np.abs(orig - rest))
    max_err = max(max_err, err)

print(f"\n✓ Manifold round-trip verified")
print(f"  Max position error : {max_err:.2e}")
assert max_err < 1e-10, f"Round-trip error too large: {max_err}"

In [ ]:
# ── Verify vocabulary ─────────────────────────────────────────────
loaded_entries = VocabularyStore.load(VOCAB_PATH)
print(f"Vocabulary verification:")
print(f"  Entries loaded     : {len(loaded_entries):,}")

# Show a sample of entries
print(f"\n  Sample entries (first 20):")
for entry in loaded_entries[:20]:
    print(f"    [{entry.register:>7s}] [{entry.rhythm:>6s}] {entry.text[:60]}")

## 7. Evaluation — query the grown manifold

In [ ]:
# ── Load vocabulary into the renderer ─────────────────────────────
pipeline._renderer.matcher.load_vocabulary(loaded_entries)
print(f"✓ Loaded {len(loaded_entries):,} vocabulary entries into C7 renderer")

In [ ]:
# ── Run sample queries ────────────────────────────────────────────
# Pick some concepts that should be on the manifold now.

all_labels = list(pipeline.manifold._points.keys())
vocab_labels = [l for l in all_labels if l.startswith("vocab::")]
print(f"Vocabulary concepts on manifold: {len(vocab_labels):,}")
print(f"Total concepts on manifold     : {len(all_labels):,}")

# Sample some vocab words for querying
sample_words = vocab_labels[:10] if len(vocab_labels) >= 10 else vocab_labels
print(f"\nSample queries using manifold concepts:")
print("=" * 70)

for label in sample_words:
    vec = pipeline.manifold.position(label)
    result = pipeline.query(vec, label=label)
    word = label.replace("vocab::", "")
    print(f"\n  Query: {word}")
    print(f"  Steps: {result.n_steps}  |  Reason: {result.termination_reason}")
    print(f"  Confidence: {result.confidence:.3f}  |  Wave: {result.wave_confidence:.3f}")
    print(f"  Output: {result.text[:120]}")
    print("-" * 70)

In [ ]:
# ── Run evaluation suite ──────────────────────────────────────────
from src.phase5 import PipelineEvaluator

evaluator = PipelineEvaluator(pipeline)

# Pick diverse concepts for evaluation
n_eval = min(50, len(vocab_labels))
eval_labels = vocab_labels[:n_eval]
eval_vecs = [pipeline.manifold.position(l) for l in eval_labels]

print(f"Running evaluation suite with {n_eval} queries...")
t0 = time.time()
suite = evaluator.run_suite(eval_vecs, eval_labels)
elapsed = time.time() - t0

print(f"\n✓ Evaluation complete ({elapsed:.1f}s)")
print(f"  Mean coherence       : {suite.mean_coherence:.4f}")
print(f"  Novelty is decaying  : {suite.novelty_is_decaying}")
print(f"  Termination reasons  :")
for reason, count in suite.termination_distribution.items():
    print(f"    {reason:30s} : {count}")

## 8. Summary statistics

In [ ]:
# ── Final summary ─────────────────────────────────────────────────
print("=" * 70)
print("  FLOW — Kaggle Vocabulary Growth — Final Report")
print("=" * 70)
print(f"")
print(f"  Corpora")
print(f"    Wikipedia          : {source_counts.get('wikipedia', 0):,} articles")
print(f"    OpenAssistant      : {source_counts.get('openassistant', 0):,} messages")
print(f"    Alpaca (cleaned)   : {source_counts.get('alpaca', 0):,} instructions")
print(f"    Glaive Functions   : {source_counts.get('glaive_functions', 0):,} tool-call examples")
print(f"    SlimOrca           : {source_counts.get('slimorca', 0):,} reasoning chains")
print(f"    Total texts        : {len(all_texts):,}")
print(f"    Total tokens       : {builder.n_tokens_fed:,}")
print(f"")
print(f"  Vocabulary")
print(f"    Words placed on M(t): {builder.n_words_placed:,}")
print(f"    Contrast judgments   : {builder.n_judgments_applied:,}")
print(f"    Expression entries   : {n_entries:,}")
print(f"    vocabulary.npz size  : {os.path.getsize(VOCAB_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Manifold")
print(f"    Total concepts      : {pipeline.n_concepts:,}")
print(f"    Dimension           : {pipeline.manifold.DIM}")
print(f"    Temperature         : {pipeline.temperature:.6f}")
print(f"    manifold.npz size   : {os.path.getsize(MANIFOLD_PATH)/1024/1024:.2f} MB")
print(f"")
print(f"  Evaluation")
print(f"    Mean coherence      : {suite.mean_coherence:.4f}")
print(f"    Novelty decaying    : {suite.novelty_is_decaying}")
print(f"")
print(f"  Artifacts saved to /kaggle/working/:")
print(f"    vocabulary_large.npz")
print(f"    manifold_large.npz")
print("=" * 70)
print(f"")
print(f"  Next: Upload artifacts to HuggingFace Hub or download for local use.")
print(f"  To reload locally:")
print(f"    pipeline = GEOPipeline.load('manifold_large.npz',")
print(f"                               vocabulary_path='vocabulary_large.npz')")

## 9. (Optional) Download artifacts

Kaggle saves everything in `/kaggle/working/` as output.  
After the notebook finishes, download from the Kaggle output tab:
- `vocabulary_large.npz`
- `manifold_large.npz`

Or push directly to HuggingFace Hub:

In [ ]:
# ── (Optional) Push to HuggingFace Hub ────────────────────────────
# Uncomment and set your token to push artifacts.
#
# !pip install huggingface_hub -q
# from huggingface_hub import HfApi
# api = HfApi()
#
# REPO_ID = "United-Visions/flow-geometric-manifold"
# api.create_repo(REPO_ID, exist_ok=True)
#
# api.upload_file(
#     path_or_fileobj=VOCAB_PATH,
#     path_in_repo="vocabulary_large.npz",
#     repo_id=REPO_ID,
# )
# api.upload_file(
#     path_or_fileobj=MANIFOLD_PATH,
#     path_in_repo="manifold_large.npz",
#     repo_id=REPO_ID,
# )
# print(f"✓ Artifacts pushed to https://huggingface.co/{REPO_ID}")